# ネットワーク

本章では、AWSソリューションアーキテクトアソシエイト試験に必要なネットワークサービスについて学習します。

## 本章の構成

- Elastic Load Balancing (ELB)
- AWS Global Accelerator
- AWS PrivateLink

### 本章のサービス一覧

| | サービス | ひとこと |
|---|----------|----------|
| <img src="images/aws-icons/vpc.png" width="32" alt="Amazon VPC"/> | **Amazon VPC** | 論理的に隔離した仮想ネットワーク |
| <img src="images/aws-icons/nat-gateway.png" width="32" alt="NAT Gateway"/> | **NAT Gateway** | プライベートサブネットから外向き通信 |
| <img src="images/aws-icons/route53.png" width="32" alt="Amazon Route 53"/> | **Amazon Route 53** | DNS・ヘルスチェック・ルーティング |
| <img src="images/aws-icons/elb.png" width="32" alt="Elastic Load Balancing"/> | **Elastic Load Balancing** | トラフィック分散（ALB/NLB） |
| <img src="images/aws-icons/cloudfront.png" width="32" alt="Amazon CloudFront"/> | **Amazon CloudFront** | コンテンツのキャッシュ配信（CDN） |
| <img src="images/aws-icons/global-accelerator.png" width="32" alt="AWS Global Accelerator"/> | **AWS Global Accelerator** | グローバル経路最適化・固定IP |
| <img src="images/aws-icons/private-link.png" width="32" alt="AWS PrivateLink"/> | **AWS PrivateLink** | VPCからサービスへプライベート接続 |
| <img src="images/aws-icons/transit-gateway.png" width="32" alt="AWS Transit Gateway"/> | **AWS Transit Gateway** | 多数VPCをハブで接続 |

> ### 試験補足：ネットワーク基盤（要点）
>
>

> 本章の主題はELB等ですが、SAAでは以下もセットで問われます。
>
>

> | 要素 | 役割 |
> |------|------|
> | **VPC** | 論理的に隔離した仮想ネットワーク |
> | **サブネット** | VPC内のIP範囲（パブリック / プライベート） |
> | **インターネットゲートウェイ（IGW）** | VPCとインターネットの接続 |
> | **NAT Gateway** | プライベートサブネットから**外向き**インターネット接続 |
> | **セキュリティグループ（SG）** | ENI単位・ステートフル（許可ルール） |
> | **ネットワークACL（NACL）** | サブネット単位・ステートレス |
> | **Route 53** | DNS・ヘルスチェック・ルーティングポリシー |
> | **Site-to-Site VPN / Direct Connect** | オンプレミスとAWSの接続 |
## Elastic Load Balancing (ELB)

**この章で覚えること**

- **ALB**＝レイヤ7、**NLB**＝レイヤ4
- 新規は ALB/NLB（CLB はレガシー）

Elastic Load Balancing (ELB) は、AWS上でのトラフィック分散サービスです。ELBは、複数のEC2インスタンスやその他のターゲットにトラフィックを自動的に分散し、高可用性と耐障害性を確保します。

ELB について説明する前に、**OSI基本参照モデル**（通信を7層に分けたモデル）を押さえます。ELBは主に**レイヤ4（TCP/UDP）**と**レイヤ7（HTTP/HTTPS）**で動作します。

| 層 | 名称 | 例 |
|----|------|-----|
| 7 | アプリケーション | HTTP, HTTPS |
| 4 | トランスポート | TCP, UDP |
| 3 | ネットワーク | IP |

ELB には、以下の3つの主要なタイプがあり、それぞれ動作する層が異なっています。

- **Application Load Balancer (ALB)**：レイヤ7（HTTP/HTTPS）。URLルーティング、WebSocket、ヘルスチェック
- **Network Load Balancer (NLB)**：レイヤ4（TCP/UDP）。超低遅延、静的IP、高スループット
- **Classic Load Balancer (CLB)**：旧式。新規は ALB/NLB

| タイプ | 向く例 |
|--------|--------|
| ALB | HTTPルーティング、マイクロサービス、コンテナ（ECS/EKS） |
| NLB | 超低遅延TCP/UDP、固定IP、ゲーム等 |
| CLB | レガシー維持のみ |

ELBの重要な機能：**ヘルスチェック**（異常インスタンスへ振らない）、**SSL/TLS終端**、**スティッキーセッション**（同一ユーザーを同じインスタンスへ）、**Auto Scaling**との連携です。

> **Route 53 との連携（試験頻出）**：DNSのエイリアスレコードでALB/NLBを指し、Route 53のヘルスチェックと組み合わせてフェイルオーバー構成にできます。
>
>

> - **ヘルスチェック**: 各ターゲットの正常性を監視し、不具合があればトラフィックを振り分けない機能
> - **SSL/TLS終端**: 安全な通信のための暗号化と復号化の処理を行う機能
---

## AWS Global Accelerator

**この章で覚えること**

- グローバル経路最適化
- 固定 IP（キャッシュは CloudFront）

**AWS Global Accelerator**は、AWSグローバルネットワークでトラフィックを最適ルーティングするサービスです。

- 複数リージョンへのトラフィック分散、障害時の自動切替
- 静的 IP の提供
- 低遅延（エッジ経由）

> **CloudFront との違い（試験頻出）**
>
>

> | サービス | 主な役割 |
> |----------|----------|
> | **CloudFront** | コンテンツの**キャッシュ配信**（CDN） |
> | **Global Accelerator** | AWSグローバルネットワークによる**経路最適化**・固定IP・障害時の切替 |

- **向く例**：グローバルユーザー向けに安定したIPと低遅延経路が欲しい
- **向かない例**：静的ファイルのキャッシュ配信のみ → CloudFront

---

## AWS PrivateLink

**この章で覚えること**

- VPC エンドポイント（Gateway / Interface）でプライベート接続

**AWS PrivateLink**は、インターネットを出さずに AWS/SaaS へプライベート接続するサービスです。

- Interface 型 VPC エンドポイント経由でプライベート IP アクセス
- セキュリティ向上、レイテンシ低減

**VPCエンドポイントとの関係**

- **Gateway型エンドポイント**：S3・DynamoDB 向け（ルートテーブルでプライベート接続）
- **Interface型エンドポイント**：PrivateLink 経由で多くのAWS/サードパーティサービスへプライベート接続

| 接続方式 | 向く例 |
|----------|--------|
| **VPC Peering** | 2つのVPC間をプライベート接続（CIDRが重複しないこと） |
| **Transit Gateway** | 多数のVPC・オンプレをハブで接続 |
| **PrivateLink** | AWSサービスやSaaSへ一方向のプライベート接続 |

- **向く例**：VPC内からSaaSやAWS APIへインターネットを出さず接続したい
- **向かない例**：パブリックインターネット経由で十分な開発用API

> ### 試験で押さえる設計の考え方（本章関連）
>
>

> - **可用性**：マルチAZ + ELB + Route 53フェイルオーバー
> - **セキュリティ**：プライベートサブネット + NAT Gateway + VPCエンドポイント
> - **パフォーマンス**：CloudFront（キャッシュ） / Global Accelerator（経路）
## まとめ

- **VPC / NAT / SG / NACL / Route 53**：ネットワーク基盤（試験補足）
- **ALB / NLB / CLB**：レイヤ7 vs 4 の負荷分散。新規は ALB/NLB
- **Global Accelerator**：グローバル経路最適化・固定IP（CDNは CloudFront）
- **PrivateLink / VPCエンドポイント / Transit Gateway**：プライベート接続


## 復習クイズ

> **取り組み方**
> 1. 下の「問題」だけを読み、口頭またはメモで答える
> 2. すべて答えたあと、**区切りセル**まで進み、確認してから**解答・解説セル**へ進む
> 3. 解答が見えている場合は、紙などで隠してから取り組む

### 問題

**Q1.** HTTPのパス（例：`/api/*`）で振り分けたい。レイヤ7で動作するロードバランサーの種類は何か。

**Q2.** 超低遅延のTCP負荷分散と、固定IPが必要。適切なロードバランサーの種類は何か。

**Q3.** 新規設計では避け、ALBやNLBを選ぶことが推奨される旧式のロードバランサーは何か。

**Q4.** 世界中のユーザーに静的コンテンツ（画像・CSS）をキャッシュして配信したい。CDNとして使うサービスは何か。

**Q5.** グローバルユーザー向けに、AWSのグローバルネットワークで経路を最適化し、固定IPと障害時の自動切替が欲しい。サービス名は何か。

**Q6.** VPC内のEC2から、インターネットを経由せずS3にアクセスしたい。S3向けに使うVPCエンドポイントの種類（Gateway / Interface）はどちらか。

**Q7.** プライベートサブネット内のEC2から、セキュリティパッチ取得のためにインターネットへ出たい。使うべきサービスは何か。

**Q8.** 障害時にスタンバイリージョンのALBへDNSで切り替えたい。DNSとヘルスチェックに使うサービスは何か。


---

## ここまでで回答を考えてください

すべての問題（Q1〜Q8）に、口頭またはメモで答えてから、**次のセル**へ進んでください。

解答・解説は次のセルにあります。まだ答えを考えていない場合は、下にスクロールしないでください。

---


## 復習クイズ　解答・解説

**A1.** **Application Load Balancer（ALB）**  
レイヤ7、HTTP/HTTPS、パスベースルーティング。→ Elastic Load Balancing

**A2.** **Network Load Balancer（NLB）**  
レイヤ4、TCP/UDP、低遅延・固定IP。→ Elastic Load Balancing

**A3.** **Classic Load Balancer（CLB）**  
レガシー向け。新規はALB/NLB。→ Elastic Load Balancing

**A4.** **Amazon CloudFront**  
コンテンツのキャッシュ配信（CDN）。→ 補足（本章外だが試験頻出）

**A5.** **AWS Global Accelerator**  
経路最適化・固定IP・マルチリージョンの切替。キャッシュはCloudFrontの役割。→ AWS Global Accelerator

**A6.** **Gateway型**  
S3・DynamoDB向け。Interface型はPrivateLink経由で多くのサービスに接続。→ AWS PrivateLink

**A7.** **NAT Gateway**  
プライベートサブネットから外向きインターネット接続。→ 試験補足：ネットワーク基盤

**A8.** **Amazon Route 53**  
DNS・ヘルスチェック・フェイルオーバールーティング。→ 試験補足：ネットワーク基盤
